In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
    cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    make_scorer,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score
)

RANDOM_STATE = 42

In [ ]:
DATA_PATH = Path("../Data/clean_data.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nTarget distribution:")
print(df["279"].value_counts())

df.head()

In [ ]:
TARGET_COLUMN = "279"

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN].astype(int)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts())

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

f1_scorer = make_scorer(
    f1_score,
    pos_label=1,
    zero_division=0
)

print("Cross-validation setup complete.")

In [ ]:
# Random Forest pipeline
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE))
])

rf_params = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5]
}

rf_search = GridSearchCV(
    rf_pipeline,
    rf_params,
    cv=cv,
    scoring=f1_scorer,
    n_jobs=-1,
    return_train_score=False
)

rf_search.fit(X, y)

print("Best Random Forest parameters:")
print(rf_search.best_params_)
print("Best Random Forest F1:", round(rf_search.best_score_, 4))

In [ ]:
# RBF SVM pipeline
svm_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", SVC(kernel="rbf"))
])

svm_params = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__gamma": ["scale", 0.01, 0.1, 1]
}

svm_search = GridSearchCV(
    svm_pipeline,
    svm_params,
    cv=cv,
    scoring=f1_scorer,
    n_jobs=-1,
    return_train_score=False
)

svm_search.fit(X, y)

print("Best SVM parameters:")
print(svm_search.best_params_)
print("Best SVM F1:", round(svm_search.best_score_, 4))

In [ ]:
tuning_results = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Baseline F1": 0.8126,
        "Tuned F1": rf_search.best_score_,
        "Best Parameters": str(rf_search.best_params_)
    },
    {
        "Model": "RBF SVM",
        "Baseline F1": 0.7059,
        "Tuned F1": svm_search.best_score_,
        "Best Parameters": str(svm_search.best_params_)
    }
])

tuning_results.to_csv("tuning_results.csv", index=False)

tuning_results

In [ ]:
best_model = rf_search.best_estimator_

# Predicted classes for each held-out fold
y_pred = cross_val_predict(
    best_model,
    X,
    y,
    cv=cv,
    method="predict",
    n_jobs=-1
)

# Predicted probabilities for the positive class (Arrhythmia = 1)
y_prob = cross_val_predict(
    best_model,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("Predictions generated successfully!")

In [ ]:
cm = confusion_matrix(y, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal (0)", "Arrhythmia (1)"]
)

disp.plot(cmap="Blues", values_format="d")
plt.title("Random Forest - Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(
    y,
    y_pred,
    target_names=["Normal", "Arrhythmia"],
    zero_division=0
))

In [ ]:
fpr, tpr, thresholds = roc_curve(y, y_prob)
roc_auc = roc_auc_score(y, y_prob)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f"Random Forest (AUC = {roc_auc:.3f})"
)

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(IMAGE_DIR / "random_forest_roc_curve.png",
            dpi=300, bbox_inches="tight")
plt.show()

print("ROC-AUC:", round(roc_auc, 4))

In [ ]:
IMAGE_DIR = Path("../Image")
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal (0)", "Arrhythmia (1)"]
)

disp.plot(cmap="Blues", values_format="d")
plt.title("Random Forest - Confusion Matrix")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "random_forest_confusion_matrix.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
final_metrics = pd.DataFrame([{
    "Model": "Tuned Random Forest",
    "Accuracy": 0.8442,
    "Precision": 0.8430,
    "Recall": 0.7838,
    "F1 Score": 0.8126,
    "ROC AUC": 0.9012
}])

final_metrics.to_csv("final_model_metrics.csv", index=False)

final_metrics